# Inspecting Feature Distributions Before Modeling

This notebook helps you inspect what your features actually look like before model training.

You will check:

- Numerical spread, skewness, and outliers
- Categorical frequency balance and rare levels
- Distribution differences by target class
- Scaling and transformation needs

Inspect first. Model second.

## Learning Objectives

By the end of this notebook, you should be able to:

1. Profile numerical and categorical distributions systematically.
2. Detect outliers, skewness, rare categories, and potential data-quality issues.
3. Compare feature behavior across target classes.
4. Decide preprocessing actions with explicit evidence.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from src.config import Config
from src.data_preprocessing import load_data, clean_data

config = Config()

TARGET_COLUMN = config.TARGET_COLUMN
NUMERICAL_FEATURES = list(config.NUMERICAL_COLUMNS)
CATEGORICAL_FEATURES = list(config.CATEGORICAL_COLUMNS)

print('Target:', TARGET_COLUMN)
print('Numerical features:', NUMERICAL_FEATURES)
print('Categorical features:', CATEGORICAL_FEATURES)

In [ ]:
df_raw = load_data(config.DATA_PATH)
df = clean_data(
    df_raw,
    numeric_columns=NUMERICAL_FEATURES,
    categorical_columns=CATEGORICAL_FEATURES,
    target_column=TARGET_COLUMN,
)

print('Raw shape:', df_raw.shape)
print('Clean shape:', df.shape)
print('Dtypes:')
print(df[NUMERICAL_FEATURES + CATEGORICAL_FEATURES + [TARGET_COLUMN]].dtypes)
print('\nTarget distribution:')
print(df[TARGET_COLUMN].value_counts(dropna=False))

## 1) Numerical Summary Statistics

Check range, center, spread, and tail behavior first.

In [ ]:
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
numeric_summary = df[NUMERICAL_FEATURES].describe(percentiles=percentiles).T
numeric_summary['skew'] = df[NUMERICAL_FEATURES].skew(numeric_only=True)
numeric_summary['kurtosis'] = df[NUMERICAL_FEATURES].kurtosis(numeric_only=True)
numeric_summary

## 2) Outlier Screening via IQR

Uses the common 1.5 x IQR rule as a first-pass diagnostic.

In [ ]:
def iqr_outlier_report(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (data[col] < lower) | (data[col] > upper)
        count = int(mask.sum())
        rows.append({
            'feature': col,
            'q1': q1,
            'q3': q3,
            'iqr': iqr,
            'lower_bound': lower,
            'upper_bound': upper,
            'outlier_count': count,
            'outlier_pct': round((count / len(data)) * 100, 2),
        })
    return pd.DataFrame(rows).sort_values('outlier_pct', ascending=False)

outlier_report = iqr_outlier_report(df, NUMERICAL_FEATURES)
outlier_report

## 3) Visual Inspection (Histograms and Boxplots)

If matplotlib is unavailable, the notebook will skip plots and continue with numeric diagnostics.

In [ ]:
HAS_MATPLOTLIB = True
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    HAS_MATPLOTLIB = False
    print('Matplotlib not available; skipping plots.')
    print(f'Import error: {exc}')

if HAS_MATPLOTLIB:
    n = len(NUMERICAL_FEATURES)
    fig, axes = plt.subplots(nrows=n, ncols=2, figsize=(12, 4 * n))

    if n == 1:
        axes = np.array([axes])

    for idx, col in enumerate(NUMERICAL_FEATURES):
        ax_hist = axes[idx, 0]
        ax_box = axes[idx, 1]

        df[col].hist(bins=30, ax=ax_hist)
        ax_hist.set_title(f'Histogram: {col}')
        ax_hist.set_xlabel(col)
        ax_hist.set_ylabel('Frequency')

        ax_box.boxplot(df[col].dropna(), vert=True)
        ax_box.set_title(f'Boxplot: {col}')
        ax_box.set_xticks([1])
        ax_box.set_xticklabels([col])

    plt.tight_layout()
    plt.show()

## 4) Categorical Distribution Inspection

Review class balance, unexpected values, and rare levels before encoding.

In [ ]:
def categorical_report(data: pd.DataFrame, columns: list[str], rare_threshold: float = 0.01) -> dict[str, pd.DataFrame]:
    reports = {}
    for col in columns:
        vc = data[col].value_counts(dropna=False)
        vf = data[col].value_counts(dropna=False, normalize=True).mul(100).round(2)
        report = pd.DataFrame({'count': vc, 'pct': vf})
        report['is_rare'] = report['pct'] < (rare_threshold * 100)
        reports[col] = report
    return reports

cat_reports = categorical_report(df, CATEGORICAL_FEATURES, rare_threshold=0.01)
for c in CATEGORICAL_FEATURES:
    print('\n' + '=' * 60)
    print(f'Categorical feature: {c}')
    print(cat_reports[c])

rare_summary_rows = []
for c in CATEGORICAL_FEATURES:
    n_rare_levels = int(cat_reports[c]['is_rare'].sum())
    rare_summary_rows.append({'feature': c, 'rare_levels_lt_1pct': n_rare_levels})

pd.DataFrame(rare_summary_rows)

## 5) Compare Distributions by Target Class

This gives early intuition for feature signal strength before modeling.

In [ ]:
grouped_summary = (
    df.groupby(TARGET_COLUMN)[NUMERICAL_FEATURES]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .transpose()
)
grouped_summary

In [ ]:
if HAS_MATPLOTLIB:
    n = len(NUMERICAL_FEATURES)
    fig, axes = plt.subplots(nrows=1, ncols=n, figsize=(5 * n, 4))

    if n == 1:
        axes = [axes]

    labels = list(df[TARGET_COLUMN].astype(str).unique())
    for idx, col in enumerate(NUMERICAL_FEATURES):
        series_by_class = [df.loc[df[TARGET_COLUMN] == lab, col].dropna() for lab in labels]
        axes[idx].boxplot(series_by_class, tick_labels=labels)
        axes[idx].set_title(f'{col} by {TARGET_COLUMN}')
        axes[idx].set_xlabel(TARGET_COLUMN)
        axes[idx].set_ylabel(col)

    plt.tight_layout()
    plt.show()

## 6) Scaling and Transformation Diagnostics

Use numeric evidence to decide scaling or transformations instead of applying them by habit.

In [ ]:
range_report = df[NUMERICAL_FEATURES].agg(['min', 'max', 'mean', 'std']).T
range_report['range'] = range_report['max'] - range_report['min']
range_report['abs_skew'] = df[NUMERICAL_FEATURES].skew(numeric_only=True).abs()
range_report = range_report.sort_values('range', ascending=False)
range_report

In [ ]:
recommendations = []
for col in NUMERICAL_FEATURES:
    skew = float(df[col].skew())
    min_val = float(df[col].min())
    outlier_pct = float(outlier_report.loc[outlier_report['feature'] == col, 'outlier_pct'].iloc[0])

    action = []
    if abs(skew) > 1.0 and min_val >= 0:
        action.append('consider log1p transform')
    if outlier_pct > 5.0:
        action.append('review capping/winsorization policy')
    if not action:
        action.append('no immediate transformation needed')

    recommendations.append({
        'feature': col,
        'skew': round(skew, 3),
        'outlier_pct': outlier_pct,
        'recommendation': '; '.join(action),
    })

recommendation_df = pd.DataFrame(recommendations)
recommendation_df

## 7) Documentation Block (Fill and Save with Your Findings)

Use the template below to record your distribution analysis decisions.

In [ ]:
distribution_notes = {
    'numerical_observations': [
        'Example: TotalCharges is right-skewed; check whether log1p improves symmetry.',
        'Example: tenure has moderate spread with manageable outliers.',
    ],
    'categorical_observations': [
        'Example: Contract has class imbalance (month-to-month dominant).',
        'Example: PaymentMethod has no rare categories below 1%.',
    ],
    'target_comparison_observations': [
        'Example: churn class shows higher MonthlyCharges median.',
        'Example: retained class shows higher tenure median.',
    ],
    'preprocessing_decisions': [
        'Use StandardScaler in preprocessing pipeline for numeric columns.',
        'Keep one-hot encoding for configured categorical columns.',
    ],
}

pd.Series(distribution_notes).to_string()

## Final Checklist

Before training, confirm:

- Numerical distributions inspected (summary + visualization)
- Outliers quantified and reviewed
- Categorical frequencies and rare levels checked
- Feature behavior compared by target class
- Scaling and transform decisions justified by evidence
- Observations documented for traceability

If these are complete, your modeling stage starts on a reliable foundation.